# Fixed DNN Training, OOF Evaluation and Results Tables - All Families



## 1. Imports and project paths



In [1]:
from pathlib import Path
from datetime import datetime
import gc
import hashlib
import json
import math
import os
import shutil
import time

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_curve,
    precision_recall_curve,
    auc,
)

import tensorflow as tf
from tensorflow.keras.backend import clear_session
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter


PROJECT_ROOT = Path.cwd()
SPLIT_DIR = PROJECT_ROOT / "data" / "split"
STATS_DIR = PROJECT_ROOT / "Stats"

STATS_DIR.mkdir(parents=True, exist_ok=True)

assert SPLIT_DIR.exists(), f"Split folder not found: {SPLIT_DIR}"

print("Project root :", PROJECT_ROOT)
print("Split folder :", SPLIT_DIR)
print("Stats folder :", STATS_DIR)
print("TensorFlow   :", tf.__version__)

# Avoid TensorFlow reserving all GPU memory at once when supported.
try:
    for gpu in tf.config.list_physical_devices("GPU"):
        tf.config.experimental.set_memory_growth(gpu, True)
except Exception as exc:
    print("GPU memory-growth configuration skipped:", exc)


C:\Users\riskf\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\riskf\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\riskf\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Pyt

Project root : c:\Users\riskf\OneDrive\A-DTI2026
Split folder : c:\Users\riskf\OneDrive\A-DTI2026\data\split
Stats folder : c:\Users\riskf\OneDrive\A-DTI2026\Stats
TensorFlow   : 2.20.0


## 2. Frozen protocol and experiment registry

In [3]:
FAMILIES = {
    "enzyme": {
        "display": "Enzyme",
        "sheet": "Enzyme",
        "batch_size": 256,
    },
    "gpcr": {
        "display": "GPCR",
        "sheet": "GPCR",
        "batch_size": 128,
    },
    "ion_channel": {
        "display": "Ion Channel",
        "sheet": "IonChannel",
        "batch_size": 128,
    },
    "nuclear_receptor": {
        "display": "Nuclear Receptor",
        "sheet": "NuclearRec",
        "batch_size": 32,
    },
}

BACKBONES = ["MSCMF", "NNMF", "LMF"]
N_OUTER_FOLDS = 5
RUNS_PER_CONFIGURATION = 10
FEATURE_DIM = 100
LATENT_DIM_PER_ENTITY = 50

TARGET_RATIOS = {
    "50_50": (1, 1),
    "80_20": (4, 1),
    "85_15": (17, 3),
    "90_10": (9, 1),
}

RATIO_DISPLAY = {
    "50_50": "50/50",
    "80_20": "80/20",
    "85_15": "85/15",
    "90_10": "90/10",
    "NO_DA": "No aug.",
}

CLASSICAL_MSCMF_METHODS = [
    "RandomOver",
    "RandomUnder",
    "TomekLinks",
    "CNNTomekLinks",
    "SMOTE",
    "SMOTETomekLinks",
    "BasicSMOTE",
    "ADASYN",
    "KMeansSMOTE",
    "LatentSampling",
    "NoiseInjection",
]

STEP_09_10_CONFIGS = (
    [("MSCMF", method) for method in CLASSICAL_MSCMF_METHODS]
    + [
        (backbone, method)
        for backbone in BACKBONES
        for method in ["VAE", "WGANGP"]
    ]
    + [("MSCMF", "LatentDiffusion")]
)

assert len(STEP_09_10_CONFIGS) == 18

# Display names are aligned with the manuscript table terminology.
DISPLAY_NAME = {
    ("MSCMF", "WGANGP"): "MSCMF (WGAN-GP)",
    ("MSCMF", "VAE"): "MSCMF (VAE)",
    ("NNMF", "WGANGP"): "NNMF (WGAN-GP)",
    ("LMF", "WGANGP"): "LMF (WGAN-GP)",
    ("NNMF", "VAE"): "NNMF (VAE)",
    ("LMF", "VAE"): "LMF (VAE)",
    ("MSCMF", "LatentDiffusion"): "Diffusion Model",
    ("MSCMF", "ADASYN"): "ADASYN",
    ("MSCMF", "LatentSampling"): "Latent Sampling",
    ("MSCMF", "NoiseInjection"): "Noise Injection",
    ("MSCMF", "RandomOver"): "Random Over-Sampling",
    ("MSCMF", "SMOTE"): "SMOTE",
    ("MSCMF", "BasicSMOTE"): "Basic SMOTE",
    ("MSCMF", "KMeansSMOTE"): "k-Means SMOTE",
    ("MSCMF", "SMOTETomekLinks"): "SMOTE + Tomek Links",
    ("MSCMF", "CNNTomekLinks"): "CNN + Tomek Links",
    ("MSCMF", "RandomUnder"): "Random Under-Sampling",
    ("MSCMF", "TomekLinks"): "Tomek Links",
}

# This ordering follows the existing manuscript table organization where possible,
# while omitting methods that are not in the frozen Step 09-10 manifest.
REPORT_METHOD_ORDER = [
    ("MSCMF", "WGANGP"),
    ("MSCMF", "VAE"),
    ("NNMF", "WGANGP"),
    ("LMF", "WGANGP"),
    ("NNMF", "VAE"),
    ("LMF", "VAE"),
    ("MSCMF", "LatentDiffusion"),
    ("MSCMF", "ADASYN"),
    ("MSCMF", "LatentSampling"),
    ("MSCMF", "NoiseInjection"),
    ("MSCMF", "RandomOver"),
    ("MSCMF", "SMOTE"),
    ("MSCMF", "BasicSMOTE"),
    ("MSCMF", "KMeansSMOTE"),
    ("MSCMF", "SMOTETomekLinks"),
    ("MSCMF", "CNNTomekLinks"),
    ("MSCMF", "RandomUnder"),
    ("MSCMF", "TomekLinks"),
]

assert set(REPORT_METHOD_ORDER) == set(STEP_09_10_CONFIGS)


def make_config_id(backbone, method, ratio_key):
    return f"{backbone}__{method}__{ratio_key}"


REPORT_CONFIGS = [
    {
        "configuration_order": 0,
        "configuration_id": "MSCMF__NoDA__NO_DA",
        "method": "MSCMF (No DA)",
        "backbone": "MSCMF",
        "internal_method": "NoDA",
        "ratio_key": "NO_DA",
        "ratio": "No aug.",
        "is_baseline": True,
    }
]

order = 1
for backbone, method in REPORT_METHOD_ORDER:
    for ratio_key in TARGET_RATIOS:
        REPORT_CONFIGS.append({
            "configuration_order": order,
            "configuration_id": make_config_id(backbone, method, ratio_key),
            "method": DISPLAY_NAME[(backbone, method)],
            "backbone": backbone,
            "internal_method": method,
            "ratio_key": ratio_key,
            "ratio": RATIO_DISPLAY[ratio_key],
            "is_baseline": False,
        })
        order += 1

assert len(REPORT_CONFIGS) == 73
assert len({c["configuration_id"] for c in REPORT_CONFIGS}) == 73

DNN_IMPLEMENTATION_VERSION = "dnn_oof_v1_fixed_5fold_10run_73configs"
T_CRITICAL_95_DF9 = 2.262

# Operational controls.
# Core checkpoint CSVs are persisted after EVERY completed run.
# The Excel workbook is refreshed after each completed configuration by default
# to reduce I/O overhead. Set the following to True if an Excel refresh is also
# required after every single run.
REFRESH_EXCEL_AFTER_EACH_RUN = False
REFRESH_EXCEL_AFTER_EACH_CONFIGURATION = True
FIT_VERBOSE = 0
DELETE_COMPLETED_RUN_FOLD_CHECKPOINTS = False

FAMILIES_TO_RUN = list(FAMILIES.keys())

EXPECTED_CONFIGS_PER_FAMILY = len(REPORT_CONFIGS)
EXPECTED_RUNS_PER_FAMILY = EXPECTED_CONFIGS_PER_FAMILY * RUNS_PER_CONFIGURATION
EXPECTED_TOTAL_RUN_ROWS = EXPECTED_RUNS_PER_FAMILY * len(FAMILIES)
EXPECTED_TOTAL_DNN_FITS = (
    EXPECTED_TOTAL_RUN_ROWS * N_OUTER_FOLDS
)

print("Configurations per family :", EXPECTED_CONFIGS_PER_FAMILY)
print("Runs per family           :", EXPECTED_RUNS_PER_FAMILY)
print("Total run-level rows      :", EXPECTED_TOTAL_RUN_ROWS)
print("Total DNN fits            :", EXPECTED_TOTAL_DNN_FITS)
print("DNN run seed policy       : NOT FIXED / NOT SAVED")


Configurations per family : 73
Runs per family           : 730
Total run-level rows      : 2920
Total DNN fits            : 14600
DNN run seed policy       : NOT FIXED / NOT SAVED


## 3. Paths, hashing, and fixed-role utilities


In [4]:
def file_sha256(path, chunk_size=1024 * 1024):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def text_sha256(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def pair_label_sha256(pair_ids, labels):
    pair_ids = np.asarray(pair_ids).astype(str)
    labels = np.asarray(labels).astype(np.int8)
    order = np.argsort(pair_ids)
    text = "".join(
        f"{pair_ids[i]}\t{int(labels[i])}\n"
        for i in order
    )
    return text_sha256(text)


def row_col_columns(df):
    if {"y_row", "y_col"}.issubset(df.columns):
        return "y_row", "y_col"
    if {"protein_index", "compound_index"}.issubset(df.columns):
        return "protein_index", "compound_index"
    raise AssertionError(
        "Role file must contain y_row/y_col or protein_index/compound_index."
    )


def role_file_path(family, outer_fold):
    return SPLIT_DIR / family / f"outer_fold_{outer_fold}_roles.csv"


def latent_dir(family, outer_fold, backbone):
    return (
        SPLIT_DIR
        / family
        / "latent"
        / f"outer_fold_{outer_fold}"
        / backbone.lower()
    )


def augmentation_dir(family, outer_fold, backbone, method):
    return (
        SPLIT_DIR
        / family
        / "augmentation"
        / f"outer_fold_{outer_fold}"
        / backbone.lower()
        / method
    )


AUGMENTATION_MANIFEST_FILE = SPLIT_DIR / "augmentation_9_10_manifest.json"
AUGMENTATION_RECONSTRUCTION_AUDIT_FILE = (
    STATS_DIR / "augmentation_reconstruction_audit.xlsx"
)

CHECKPOINT_ROOT = SPLIT_DIR / "dnn_11_17_checkpoints"
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

RUN_METRICS_FILE = STATS_DIR / "dnn_11_17_run_metrics_checkpoint.csv"
FOLD_AUDIT_FILE = STATS_DIR / "dnn_11_17_fold_audit_checkpoint.csv"
RESULTS_WORKBOOK = STATS_DIR / "DNN_OOF_Final_Results.xlsx"
RESULTS_WORKBOOK_TEMP = STATS_DIR / "DNN_OOF_Final_Results_TEMP.xlsx"
VALIDATION_WORKBOOK = STATS_DIR / "DNN_11_17_Validation.xlsx"
DNN_MANIFEST_FILE = SPLIT_DIR / "dnn_11_17_manifest.json"

print("Run checkpoint CSV :", RUN_METRICS_FILE)
print("Fold audit CSV     :", FOLD_AUDIT_FILE)
print("Results workbook   :", RESULTS_WORKBOOK)


Run checkpoint CSV : c:\Users\riskf\OneDrive\A-DTI2026\Stats\dnn_11_17_run_metrics_checkpoint.csv
Fold audit CSV     : c:\Users\riskf\OneDrive\A-DTI2026\Stats\dnn_11_17_fold_audit_checkpoint.csv
Results workbook   : c:\Users\riskf\OneDrive\A-DTI2026\Stats\DNN_OOF_Final_Results.xlsx


## 4. Fixed DNN architecture





In [5]:
def build_model():
    clear_session()

    custom_adam = Adam(
        learning_rate=0.001,
        beta_1=0.9,
        beta_2=0.999,
        epsilon=1e-7,
    )

    l2_reg = None

    model = Sequential([
        Dense(
            512,
            activation="relu",
            input_shape=(100,),
            kernel_regularizer=l2_reg,
        ),
        Dropout(0.1),
        Dense(
            256,
            activation="relu",
            kernel_regularizer=l2_reg,
        ),
        Dense(
            128,
            activation="relu",
            kernel_regularizer=l2_reg,
        ),
        Dense(
            2,
            activation="sigmoid",
        ),
    ])

    model.compile(
        optimizer=custom_adam,
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )

    return model


model_check = build_model()
model_check.summary()

assert model_check.input_shape[-1] == FEATURE_DIM
assert model_check.output_shape[-1] == 2

clear_session()
del model_check
gc.collect()


C:\Users\riskf\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 512)            │        51,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 216,194 (844.51 KB)

 Trainable params: 216,194 (844.51 KB)

 Non-trainable params: 0 (0.00 B)

1052

## 5. Step 09-10 and five-fold preflight




In [6]:
def validate_family_outer_roles(family):
    fold_frames = []

    for outer_fold in range(1, N_OUTER_FOLDS + 1):
        path = role_file_path(family, outer_fold)
        assert path.exists(), f"Missing role file: {path}"

        df = pd.read_csv(path)
        assert df["pair_id"].is_unique
        assert set(df["role"].unique()) == {"train", "validation", "test"}
        assert set(np.unique(df["y"].astype(int))) <= {0, 1}

        fold_frames.append(df)

    reference = fold_frames[0][["pair_id", "y"]].copy()
    reference["pair_id"] = reference["pair_id"].astype(str)
    reference["y"] = reference["y"].astype(np.int8)
    reference = reference.sort_values("pair_id").reset_index(drop=True)

    for fold, df in enumerate(fold_frames, start=1):
        check = df[["pair_id", "y"]].copy()
        check["pair_id"] = check["pair_id"].astype(str)
        check["y"] = check["y"].astype(np.int8)
        check = check.sort_values("pair_id").reset_index(drop=True)
        pd.testing.assert_frame_equal(reference, check)

    test_records = []
    for outer_fold, df in enumerate(fold_frames, start=1):
        test = df.loc[df["role"].eq("test"), ["pair_id", "y"]].copy()
        test["outer_fold"] = outer_fold
        test_records.append(test)

    all_test = pd.concat(test_records, ignore_index=True)
    assert len(all_test) == len(reference)
    assert all_test["pair_id"].is_unique
    assert set(all_test["pair_id"].astype(str)) == set(reference["pair_id"])

    expected_sha = pair_label_sha256(
        reference["pair_id"].to_numpy(),
        reference["y"].to_numpy(),
    )

    return {
        "n_total": len(reference),
        "n_positive": int((reference["y"] == 1).sum()),
        "n_non_interaction": int((reference["y"] == 0).sum()),
        "pair_label_sha256": expected_sha,
        "reference": reference,
    }


def run_preflight():
    assert AUGMENTATION_MANIFEST_FILE.exists(), (
        f"Missing Step 09-10 manifest: {AUGMENTATION_MANIFEST_FILE}"
    )

    manifest = json.loads(
        AUGMENTATION_MANIFEST_FILE.read_text(encoding="utf-8")
    )

    manifest_configs = {
        (item["backbone"], item["method"])
        for item in manifest["method_backbone_configurations"]
    }

    assert manifest.get("unique_augmentation_algorithms") == 14
    assert len(manifest_configs) == 18
    assert manifest_configs == set(STEP_09_10_CONFIGS)
    assert set(manifest["target_ratios"].keys()) == set(TARGET_RATIOS.keys())
    assert manifest.get("validation_used") is False
    assert manifest.get("outer_test_used") is False

    assert AUGMENTATION_RECONSTRUCTION_AUDIT_FILE.exists(), (
        "Missing Step 10 reconstruction audit: "
        f"{AUGMENTATION_RECONSTRUCTION_AUDIT_FILE}"
    )

    reconstruction_audit = pd.read_excel(
        AUGMENTATION_RECONSTRUCTION_AUDIT_FILE,
        sheet_name="Reconstruction Audit",
    )

    assert len(reconstruction_audit) == 1440
    assert reconstruction_audit["final_pass"].astype(bool).all()
    assert reconstruction_audit["train_only_pass"].astype(bool).all()

    family_reference = {}
    missing_files = []

    for family in FAMILIES:
        family_reference[family] = validate_family_outer_roles(family)

        for outer_fold in range(1, N_OUTER_FOLDS + 1):
            for backbone in BACKBONES:
                ldir = latent_dir(family, outer_fold, backbone)
                for filename in [
                    "protein_embeddings.npy",
                    "compound_embeddings.npy",
                    "fit_metadata.json",
                ]:
                    path = ldir / filename
                    if not path.exists():
                        missing_files.append(str(path))

            for backbone, method in STEP_09_10_CONFIGS:
                adir = augmentation_dir(
                    family,
                    outer_fold,
                    backbone,
                    method,
                )

                pool_file = adir / "synthetic_pool.npy"
                if not pool_file.exists():
                    missing_files.append(str(pool_file))

                for ratio_key in TARGET_RATIOS:
                    plan_file = adir / f"ratio_{ratio_key}_plan.npz"
                    if not plan_file.exists():
                        missing_files.append(str(plan_file))

    if missing_files:
        preview = "\n".join(missing_files[:25])
        raise FileNotFoundError(
            f"Missing {len(missing_files)} required Step 06-10 file(s).\n"
            f"First missing paths:\n{preview}"
        )

    print("PASS - Step 09-10 manifest: 14 algorithms / 18 configurations / 4 ratios.")
    print("PASS - Step 10 reconstruction audit: 1,440 / 1,440 rows pass.")
    print("PASS - All required role, latent, synthetic-pool, and ratio-plan files exist.")

    for family, info in family_reference.items():
        print(
            f"PASS - {FAMILIES[family]['display']}: "
            f"{info['n_total']:,} original pairs, outer-test coverage exactly once."
        )

    return family_reference


FAMILY_REFERENCE = run_preflight()


PASS - Step 09-10 manifest: 14 algorithms / 18 configurations / 4 ratios.
PASS - Step 10 reconstruction audit: 1,440 / 1,440 rows pass.
PASS - All required role, latent, synthetic-pool, and ratio-plan files exist.
PASS - Enzyme: 295,480 original pairs, outer-test coverage exactly once.
PASS - GPCR: 21,185 original pairs, outer-test coverage exactly once.
PASS - Ion Channel: 42,840 original pairs, outer-test coverage exactly once.
PASS - Nuclear Receptor: 1,404 original pairs, outer-test coverage exactly once.


## 6. Fold data preparation and normalization




In [7]:
def represent_roles(roles, protein_embeddings, compound_embeddings):
    row_col, col_col = row_col_columns(roles)

    p_idx = roles[row_col].astype(int).to_numpy()
    c_idx = roles[col_col].astype(int).to_numpy()

    X = np.concatenate(
        [
            protein_embeddings[p_idx],
            compound_embeddings[c_idx],
        ],
        axis=1,
    ).astype(np.float32)

    assert X.shape == (len(roles), FEATURE_DIM)
    assert np.isfinite(X).all()

    return X


def prepare_fold_data(family, outer_fold, backbone):
    roles_all = pd.read_csv(role_file_path(family, outer_fold))
    roles_all["pair_id"] = roles_all["pair_id"].astype(str)
    roles_all["y"] = roles_all["y"].astype(np.int8)

    train_roles = (
        roles_all.loc[roles_all["role"].eq("train")]
        .copy()
        .reset_index(drop=True)
    )
    validation_roles = (
        roles_all.loc[roles_all["role"].eq("validation")]
        .copy()
        .reset_index(drop=True)
    )
    test_roles = (
        roles_all.loc[roles_all["role"].eq("test")]
        .copy()
        .reset_index(drop=True)
    )

    train_ids = set(train_roles["pair_id"])
    validation_ids = set(validation_roles["pair_id"])
    test_ids = set(test_roles["pair_id"])

    assert train_ids.isdisjoint(validation_ids)
    assert train_ids.isdisjoint(test_ids)
    assert validation_ids.isdisjoint(test_ids)

    ldir = latent_dir(family, outer_fold, backbone)
    A = np.load(ldir / "protein_embeddings.npy").astype(np.float32)
    B = np.load(ldir / "compound_embeddings.npy").astype(np.float32)

    assert A.ndim == 2 and A.shape[1] == LATENT_DIM_PER_ENTITY
    assert B.ndim == 2 and B.shape[1] == LATENT_DIM_PER_ENTITY
    assert np.isfinite(A).all()
    assert np.isfinite(B).all()

    X_train_raw = represent_roles(train_roles, A, B)
    X_validation_raw = represent_roles(validation_roles, A, B)
    X_test_raw = represent_roles(test_roles, A, B)

    y_train = train_roles["y"].to_numpy(dtype=np.int8)
    y_validation = validation_roles["y"].to_numpy(dtype=np.int8)
    y_test = test_roles["y"].to_numpy(dtype=np.int8)

    scaler = StandardScaler()
    scaler.fit(X_train_raw)

    X_train_scaled = scaler.transform(X_train_raw).astype(np.float32)
    X_validation_scaled = scaler.transform(X_validation_raw).astype(np.float32)
    X_test_scaled = scaler.transform(X_test_raw).astype(np.float32)

    assert np.isfinite(X_train_scaled).all()
    assert np.isfinite(X_validation_scaled).all()
    assert np.isfinite(X_test_scaled).all()

    # Validation labels are used only for model.fit validation_data.
    y_validation_cat = to_categorical(
        y_validation,
        num_classes=2,
    ).astype(np.float32)

    del X_train_raw, X_validation_raw, X_test_raw, A, B
    gc.collect()

    return {
        "family": family,
        "outer_fold": outer_fold,
        "backbone": backbone,
        "roles_all": roles_all,
        "train_roles": train_roles,
        "validation_roles": validation_roles,
        "test_roles": test_roles,
        "X_train_scaled": X_train_scaled,
        "y_train": y_train,
        "X_validation_scaled": X_validation_scaled,
        "y_validation": y_validation,
        "y_validation_cat": y_validation_cat,
        "X_test_scaled": X_test_scaled,
        "y_test": y_test,
        "scaler": scaler,
        "scaler_fit_n": len(y_train),
        "role_file_sha256": file_sha256(role_file_path(family, outer_fold)),
        "latent_metadata_sha256": file_sha256(
            latent_dir(family, outer_fold, backbone) / "fit_metadata.json"
        ),
    }


def build_backbone_fold_cache(family, backbone):
    print(
        f"Preparing fixed scaled fold cache: "
        f"{FAMILIES[family]['display']} | {backbone}"
    )

    cache = {}
    for outer_fold in range(1, N_OUTER_FOLDS + 1):
        cache[outer_fold] = prepare_fold_data(
            family,
            outer_fold,
            backbone,
        )
        print(
            f"  fold {outer_fold}: "
            f"train={len(cache[outer_fold]['y_train']):,}, "
            f"validation={len(cache[outer_fold]['y_validation']):,}, "
            f"test={len(cache[outer_fold]['y_test']):,}"
        )

    return cache


## 7. Reconstruct the exact Step 09 training configuration



In [8]:
def assemble_scaled_training_set(
    family,
    outer_fold,
    config,
    fold_data,
):
    if config["is_baseline"]:
        X_final = fold_data["X_train_scaled"]
        y_final = fold_data["y_train"]

        info = {
            "n_train_source": len(y_final),
            "n_train_final": len(y_final),
            "n_real": len(y_final),
            "n_synthetic": 0,
            "n_non_interaction": int((y_final == 0).sum()),
            "n_interaction": int((y_final == 1).sum()),
        }
        return X_final, y_final, info

    backbone = config["backbone"]
    method = config["internal_method"]
    ratio_key = config["ratio_key"]

    assert backbone == fold_data["backbone"]

    adir = augmentation_dir(
        family,
        outer_fold,
        backbone,
        method,
    )

    plan_file = adir / f"ratio_{ratio_key}_plan.npz"
    pool_file = adir / "synthetic_pool.npy"

    with np.load(plan_file, allow_pickle=False) as plan:
        real_indices = plan["real_indices"].astype(np.int64)
        synthetic_indices = plan["synthetic_indices"].astype(np.int64)

    y_source = fold_data["y_train"]
    X_source_scaled = fold_data["X_train_scaled"]

    assert real_indices.ndim == 1
    assert synthetic_indices.ndim == 1
    assert (real_indices >= 0).all()
    assert (real_indices < len(y_source)).all()

    X_real_scaled = X_source_scaled[real_indices].astype(np.float32, copy=False)
    y_real = y_source[real_indices].astype(np.int8, copy=False)

    if len(synthetic_indices):
        synthetic_pool = np.load(
            pool_file,
            mmap_mode="r",
        )

        assert synthetic_pool.ndim == 2
        assert synthetic_pool.shape[1] == FEATURE_DIM
        assert (synthetic_indices >= 0).all()
        assert (synthetic_indices < len(synthetic_pool)).all()

        X_syn_raw = np.asarray(
            synthetic_pool[synthetic_indices],
            dtype=np.float32,
        )
        assert np.isfinite(X_syn_raw).all()

        X_syn_scaled = fold_data["scaler"].transform(
            X_syn_raw
        ).astype(np.float32)
        assert np.isfinite(X_syn_scaled).all()

        X_final = np.vstack([
            X_real_scaled,
            X_syn_scaled,
        ]).astype(np.float32, copy=False)

        y_final = np.concatenate([
            y_real,
            np.ones(len(synthetic_indices), dtype=np.int8),
        ])

        del synthetic_pool, X_syn_raw, X_syn_scaled
    else:
        X_final = np.asarray(X_real_scaled, dtype=np.float32)
        y_final = np.asarray(y_real, dtype=np.int8)

    assert X_final.shape == (len(y_final), FEATURE_DIM)
    assert np.isfinite(X_final).all()
    assert set(np.unique(y_final)) <= {0, 1}

    neg_units, pos_units = TARGET_RATIOS[ratio_key]
    n0 = int((y_final == 0).sum())
    n1 = int((y_final == 1).sum())
    assert n0 * pos_units == n1 * neg_units

    info = {
        "n_train_source": len(y_source),
        "n_train_final": len(y_final),
        "n_real": len(real_indices),
        "n_synthetic": len(synthetic_indices),
        "n_non_interaction": n0,
        "n_interaction": n1,
    }

    return X_final, y_final, info


## 8. Fold prediction checkpoints — Steps 11 and 12



In [9]:
def safe_token(text):
    return (
        str(text)
        .replace("/", "-")
        .replace("\\", "-")
        .replace(" ", "_")
        .replace("(", "")
        .replace(")", "")
    )


def run_checkpoint_dir(family, config, run_number):
    return (
        CHECKPOINT_ROOT
        / family
        / safe_token(config["configuration_id"])
        / f"run_{run_number:02d}"
    )


def fold_checkpoint_path(family, config, run_number, outer_fold):
    return (
        run_checkpoint_dir(family, config, run_number)
        / f"fold_{outer_fold}.npz"
    )


def scalar_from_npz(z, key):
    return z[key].item()


def valid_fold_checkpoint(
    path,
    family,
    config,
    run_number,
    outer_fold,
    fold_data,
):
    path = Path(path)
    if not path.exists():
        return False

    try:
        with np.load(path, allow_pickle=False) as z:
            required = {
                "implementation_version",
                "family",
                "configuration_id",
                "backbone",
                "internal_method",
                "ratio_key",
                "run",
                "outer_fold",
                "pair_id",
                "y_true",
                "y_pred",
                "y_score",
                "n_train_source",
                "n_train_final",
                "n_real",
                "n_synthetic",
                "n_train_non_interaction",
                "n_train_interaction",
                "scaler_fit_n",
                "n_validation",
                "n_test",
                "final_train_loss",
                "final_train_accuracy",
                "final_val_loss",
                "final_val_accuracy",
                "fit_seconds",
                "role_file_sha256",
                "latent_metadata_sha256",
            }

            if not required.issubset(set(z.files)):
                return False

            if scalar_from_npz(z, "implementation_version") != DNN_IMPLEMENTATION_VERSION:
                return False
            if scalar_from_npz(z, "family") != family:
                return False
            if scalar_from_npz(z, "configuration_id") != config["configuration_id"]:
                return False
            if scalar_from_npz(z, "backbone") != config["backbone"]:
                return False
            if scalar_from_npz(z, "internal_method") != config["internal_method"]:
                return False
            if scalar_from_npz(z, "ratio_key") != config["ratio_key"]:
                return False
            if int(scalar_from_npz(z, "run")) != int(run_number):
                return False
            if int(scalar_from_npz(z, "outer_fold")) != int(outer_fold):
                return False

            pair_id = z["pair_id"].astype(str)
            y_true = z["y_true"].astype(np.int8)
            y_pred = z["y_pred"].astype(np.int8)
            y_score = z["y_score"].astype(np.float64)

            expected_ids = fold_data["test_roles"]["pair_id"].astype(str).to_numpy()
            expected_y = fold_data["y_test"].astype(np.int8)

            if not np.array_equal(pair_id, expected_ids):
                return False
            if not np.array_equal(y_true, expected_y):
                return False
            if len(np.unique(pair_id)) != len(pair_id):
                return False
            if not set(np.unique(y_pred)) <= {0, 1}:
                return False
            if not np.isfinite(y_score).all():
                return False
            if ((y_score < -1e-7) | (y_score > 1 + 1e-7)).any():
                return False

            if scalar_from_npz(z, "role_file_sha256") != fold_data["role_file_sha256"]:
                return False
            if scalar_from_npz(z, "latent_metadata_sha256") != fold_data["latent_metadata_sha256"]:
                return False

        return True

    except Exception:
        return False


def train_and_save_fold(
    family,
    config,
    run_number,
    outer_fold,
    fold_data,
):
    checkpoint = fold_checkpoint_path(
        family,
        config,
        run_number,
        outer_fold,
    )
    checkpoint.parent.mkdir(parents=True, exist_ok=True)

    if valid_fold_checkpoint(
        checkpoint,
        family,
        config,
        run_number,
        outer_fold,
        fold_data,
    ):
        print(
            f"    fold {outer_fold}: valid prediction checkpoint found; resuming."
        )
        return checkpoint

    if checkpoint.exists():
        checkpoint.unlink()

    X_train, y_train, assembly = assemble_scaled_training_set(
        family,
        outer_fold,
        config,
        fold_data,
    )

    y_train_cat = to_categorical(
        y_train,
        num_classes=2,
    ).astype(np.float32)

    model = build_model()
    batch_size = FAMILIES[family]["batch_size"]

    start = time.time()

    history = model.fit(
        X_train,
        y_train_cat,
        validation_data=(
            fold_data["X_validation_scaled"],
            fold_data["y_validation_cat"],
        ),
        epochs=12,
        batch_size=batch_size,
        shuffle=True,
        verbose=FIT_VERBOSE,
    )

    fit_seconds = time.time() - start

    # Step 12: the outer-test fold is touched only after model.fit has completed.
    y_prob = model.predict(
        fold_data["X_test_scaled"],
        verbose=0,
    )

    assert y_prob.shape == (len(fold_data["y_test"]), 2)
    assert np.isfinite(y_prob).all()

    y_pred = np.argmax(y_prob, axis=1).astype(np.int8)
    y_score = y_prob[:, 1].astype(np.float32)
    y_true = fold_data["y_test"].astype(np.int8)
    pair_id = fold_data["test_roles"]["pair_id"].astype(str).to_numpy(dtype=str)

    assert set(np.unique(y_pred)) <= {0, 1}
    assert np.isfinite(y_score).all()

    final_train_loss = float(history.history["loss"][-1])
    final_train_accuracy = float(history.history["accuracy"][-1])
    final_val_loss = float(history.history["val_loss"][-1])
    final_val_accuracy = float(history.history["val_accuracy"][-1])

    for value in [
        final_train_loss,
        final_train_accuracy,
        final_val_loss,
        final_val_accuracy,
    ]:
        assert np.isfinite(value)

    np.savez_compressed(
        checkpoint,
        implementation_version=np.array(DNN_IMPLEMENTATION_VERSION),
        family=np.array(family),
        configuration_id=np.array(config["configuration_id"]),
        backbone=np.array(config["backbone"]),
        internal_method=np.array(config["internal_method"]),
        ratio_key=np.array(config["ratio_key"]),
        run=np.array(run_number, dtype=np.int16),
        outer_fold=np.array(outer_fold, dtype=np.int8),
        pair_id=pair_id,
        y_true=y_true,
        y_pred=y_pred,
        y_score=y_score,
        n_train_source=np.array(assembly["n_train_source"], dtype=np.int64),
        n_train_final=np.array(assembly["n_train_final"], dtype=np.int64),
        n_real=np.array(assembly["n_real"], dtype=np.int64),
        n_synthetic=np.array(assembly["n_synthetic"], dtype=np.int64),
        n_train_non_interaction=np.array(
            assembly["n_non_interaction"], dtype=np.int64
        ),
        n_train_interaction=np.array(
            assembly["n_interaction"], dtype=np.int64
        ),
        scaler_fit_n=np.array(fold_data["scaler_fit_n"], dtype=np.int64),
        n_validation=np.array(len(fold_data["y_validation"]), dtype=np.int64),
        n_test=np.array(len(y_true), dtype=np.int64),
        final_train_loss=np.array(final_train_loss, dtype=np.float64),
        final_train_accuracy=np.array(final_train_accuracy, dtype=np.float64),
        final_val_loss=np.array(final_val_loss, dtype=np.float64),
        final_val_accuracy=np.array(final_val_accuracy, dtype=np.float64),
        fit_seconds=np.array(fit_seconds, dtype=np.float64),
        role_file_sha256=np.array(fold_data["role_file_sha256"]),
        latent_metadata_sha256=np.array(fold_data["latent_metadata_sha256"]),
    )

    assert valid_fold_checkpoint(
        checkpoint,
        family,
        config,
        run_number,
        outer_fold,
        fold_data,
    )

    print(
        f"    fold {outer_fold}: PASS | "
        f"train={assembly['n_train_final']:,} | "
        f"validation={len(fold_data['y_validation']):,} | "
        f"test={len(y_true):,} | "
        f"{fit_seconds:.1f}s"
    )

    del model, history, X_train, y_train, y_train_cat, y_prob
    clear_session()
    gc.collect()

    return checkpoint


## 9. Full-family OOF assembly and run metrics — Steps 13 and 14


In [10]:
METRIC_COLUMNS = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC_AUC",
    "PR_AUC",
]

RUN_METRIC_COLUMNS = [
    "ImplementationVersion",
    "FamilyKey",
    "Family",
    "ConfigurationOrder",
    "ConfigurationID",
    "Method",
    "Backbone",
    "InternalMethod",
    "RatioKey",
    "Ratio",
    "Run",
    "N_Folds",
    "N_OOF",
    "N_OOF_NonInteraction",
    "N_OOF_Interaction",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC_AUC",
    "PR_AUC",
    "OOF_PairLabel_SHA256",
    "DNN_Seed_Policy",
    "Run_Fit_Seconds",
    "Status",
    "CompletedAt",
]

FOLD_AUDIT_COLUMNS = [
    "ImplementationVersion",
    "FamilyKey",
    "Family",
    "ConfigurationID",
    "Method",
    "Backbone",
    "InternalMethod",
    "RatioKey",
    "Ratio",
    "Run",
    "OuterFold",
    "N_Train_Source",
    "N_Train_Final",
    "N_Real",
    "N_Synthetic",
    "N_Train_NonInteraction",
    "N_Train_Interaction",
    "Scaler_Fit_N",
    "N_Validation",
    "N_Test",
    "Final_Train_Loss",
    "Final_Train_Accuracy",
    "Final_Val_Loss",
    "Final_Val_Accuracy",
    "Fit_Seconds",
    "Status",
]


def load_fold_checkpoint(path):
    with np.load(path, allow_pickle=False) as z:
        return {
            "pair_id": z["pair_id"].astype(str),
            "y_true": z["y_true"].astype(np.int8),
            "y_pred": z["y_pred"].astype(np.int8),
            "y_score": z["y_score"].astype(np.float64),
            "n_train_source": int(scalar_from_npz(z, "n_train_source")),
            "n_train_final": int(scalar_from_npz(z, "n_train_final")),
            "n_real": int(scalar_from_npz(z, "n_real")),
            "n_synthetic": int(scalar_from_npz(z, "n_synthetic")),
            "n_train_non_interaction": int(
                scalar_from_npz(z, "n_train_non_interaction")
            ),
            "n_train_interaction": int(
                scalar_from_npz(z, "n_train_interaction")
            ),
            "scaler_fit_n": int(scalar_from_npz(z, "scaler_fit_n")),
            "n_validation": int(scalar_from_npz(z, "n_validation")),
            "n_test": int(scalar_from_npz(z, "n_test")),
            "final_train_loss": float(scalar_from_npz(z, "final_train_loss")),
            "final_train_accuracy": float(
                scalar_from_npz(z, "final_train_accuracy")
            ),
            "final_val_loss": float(scalar_from_npz(z, "final_val_loss")),
            "final_val_accuracy": float(
                scalar_from_npz(z, "final_val_accuracy")
            ),
            "fit_seconds": float(scalar_from_npz(z, "fit_seconds")),
        }


def compute_run_metrics(y_true, y_pred, y_score):
    accuracy = accuracy_score(y_true, y_pred)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0,
    )

    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)

    precision_vals, recall_vals, _ = precision_recall_curve(
        y_true,
        y_score,
    )
    pr_auc = auc(recall_vals, precision_vals)

    metrics = {
        "Accuracy": float(accuracy),
        "Precision": float(precision),
        "Recall": float(recall),
        "F1": float(f1),
        "ROC_AUC": float(roc_auc),
        "PR_AUC": float(pr_auc),
    }

    for value in metrics.values():
        assert np.isfinite(value)
        assert -1e-12 <= value <= 1 + 1e-12

    return metrics


def assemble_run_oof_and_audit(
    family,
    config,
    run_number,
    fold_cache,
):
    pieces = []
    fold_audit_rows = []
    run_fit_seconds = 0.0

    for outer_fold in range(1, N_OUTER_FOLDS + 1):
        path = fold_checkpoint_path(
            family,
            config,
            run_number,
            outer_fold,
        )

        assert valid_fold_checkpoint(
            path,
            family,
            config,
            run_number,
            outer_fold,
            fold_cache[outer_fold],
        ), f"Invalid fold checkpoint: {path}"

        fold_result = load_fold_checkpoint(path)

        pieces.append(pd.DataFrame({
            "pair_id": fold_result["pair_id"],
            "y_true": fold_result["y_true"],
            "y_pred": fold_result["y_pred"],
            "y_score": fold_result["y_score"],
            "outer_fold": outer_fold,
        }))

        run_fit_seconds += fold_result["fit_seconds"]

        fold_audit_rows.append({
            "ImplementationVersion": DNN_IMPLEMENTATION_VERSION,
            "FamilyKey": family,
            "Family": FAMILIES[family]["display"],
            "ConfigurationID": config["configuration_id"],
            "Method": config["method"],
            "Backbone": config["backbone"],
            "InternalMethod": config["internal_method"],
            "RatioKey": config["ratio_key"],
            "Ratio": config["ratio"],
            "Run": run_number,
            "OuterFold": outer_fold,
            "N_Train_Source": fold_result["n_train_source"],
            "N_Train_Final": fold_result["n_train_final"],
            "N_Real": fold_result["n_real"],
            "N_Synthetic": fold_result["n_synthetic"],
            "N_Train_NonInteraction": fold_result["n_train_non_interaction"],
            "N_Train_Interaction": fold_result["n_train_interaction"],
            "Scaler_Fit_N": fold_result["scaler_fit_n"],
            "N_Validation": fold_result["n_validation"],
            "N_Test": fold_result["n_test"],
            "Final_Train_Loss": fold_result["final_train_loss"],
            "Final_Train_Accuracy": fold_result["final_train_accuracy"],
            "Final_Val_Loss": fold_result["final_val_loss"],
            "Final_Val_Accuracy": fold_result["final_val_accuracy"],
            "Fit_Seconds": fold_result["fit_seconds"],
            "Status": "PASS",
        })

    oof = pd.concat(pieces, ignore_index=True)
    oof["pair_id"] = oof["pair_id"].astype(str)
    oof = oof.sort_values("pair_id").reset_index(drop=True)

    reference = FAMILY_REFERENCE[family]

    assert len(oof) == reference["n_total"]
    assert oof["pair_id"].is_unique
    assert set(oof["pair_id"]) == set(reference["reference"]["pair_id"])

    oof_sha = pair_label_sha256(
        oof["pair_id"].to_numpy(),
        oof["y_true"].to_numpy(),
    )
    assert oof_sha == reference["pair_label_sha256"]

    metrics = compute_run_metrics(
        oof["y_true"].to_numpy(dtype=np.int8),
        oof["y_pred"].to_numpy(dtype=np.int8),
        oof["y_score"].to_numpy(dtype=np.float64),
    )

    run_row = {
        "ImplementationVersion": DNN_IMPLEMENTATION_VERSION,
        "FamilyKey": family,
        "Family": FAMILIES[family]["display"],
        "ConfigurationOrder": config["configuration_order"],
        "ConfigurationID": config["configuration_id"],
        "Method": config["method"],
        "Backbone": config["backbone"],
        "InternalMethod": config["internal_method"],
        "RatioKey": config["ratio_key"],
        "Ratio": config["ratio"],
        "Run": run_number,
        "N_Folds": N_OUTER_FOLDS,
        "N_OOF": len(oof),
        "N_OOF_NonInteraction": int((oof["y_true"] == 0).sum()),
        "N_OOF_Interaction": int((oof["y_true"] == 1).sum()),
        **metrics,
        "OOF_PairLabel_SHA256": oof_sha,
        "DNN_Seed_Policy": "not fixed; not saved",
        "Run_Fit_Seconds": run_fit_seconds,
        "Status": "PASS",
        "CompletedAt": datetime.now().isoformat(timespec="seconds"),
    }

    return run_row, fold_audit_rows


## 10. Atomic CSV checkpoints and restart logic

The run-level CSV is the primary durable checkpoint. A completed run is skipped only when its row has the current implementation version, `PASS` status, five folds, the expected OOF count, the expected OOF pair/label hash, and finite metrics.


In [11]:
def empty_run_metrics_df():
    return pd.DataFrame(columns=RUN_METRIC_COLUMNS)


def empty_fold_audit_df():
    return pd.DataFrame(columns=FOLD_AUDIT_COLUMNS)


def load_checkpoint_csv(path, columns):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame(columns=columns)

    df = pd.read_csv(path)
    missing = [col for col in columns if col not in df.columns]
    if missing:
        raise ValueError(
            f"Checkpoint file {path} is missing columns: {missing}"
        )

    return df[columns].copy()


def atomic_write_csv(df, path):
    path = Path(path)
    temp = path.with_name(path.stem + "_TEMP" + path.suffix)

    if temp.exists():
        try:
            temp.unlink()
        except PermissionError as exc:
            raise PermissionError(
                f"Temporary checkpoint is locked: {temp}. Close it and rerun."
            ) from exc

    df.to_csv(temp, index=False)

    try:
        os.replace(temp, path)
    except PermissionError as exc:
        raise PermissionError(
            f"Checkpoint destination is locked: {path}. "
            "Close it in Excel/File Explorer and rerun. "
            "The five fold prediction checkpoints remain available, so the run "
            "can be reconstructed without retraining completed folds."
        ) from exc


def upsert_rows(df, rows, key_columns, all_columns, path):
    incoming = pd.DataFrame(rows)
    incoming = incoming.reindex(columns=all_columns)

    if df.empty:
        out = incoming.copy()
    else:
        old_keys = pd.MultiIndex.from_frame(
            df[key_columns].astype(str)
        )
        new_keys = pd.MultiIndex.from_frame(
            incoming[key_columns].astype(str)
        )
        keep = ~old_keys.isin(new_keys)
        out = pd.concat(
            [df.loc[keep], incoming],
            ignore_index=True,
        )

    out = out.reindex(columns=all_columns)
    atomic_write_csv(out, path)
    return out


def completed_run_is_valid(run_metrics_df, family, config, run_number):
    if run_metrics_df.empty:
        return False

    rows = run_metrics_df.loc[
        run_metrics_df["FamilyKey"].astype(str).eq(family)
        & run_metrics_df["ConfigurationID"].astype(str).eq(
            config["configuration_id"]
        )
        & pd.to_numeric(run_metrics_df["Run"], errors="coerce").eq(run_number)
    ]

    if len(rows) != 1:
        return False

    row = rows.iloc[0]

    if row["ImplementationVersion"] != DNN_IMPLEMENTATION_VERSION:
        return False
    if row["Status"] != "PASS":
        return False
    if int(row["N_Folds"]) != N_OUTER_FOLDS:
        return False
    if int(row["N_OOF"]) != FAMILY_REFERENCE[family]["n_total"]:
        return False
    if row["OOF_PairLabel_SHA256"] != FAMILY_REFERENCE[family]["pair_label_sha256"]:
        return False

    values = pd.to_numeric(row[METRIC_COLUMNS], errors="coerce").to_numpy(dtype=float)
    if not np.isfinite(values).all():
        return False
    if ((values < -1e-12) | (values > 1 + 1e-12)).any():
        return False

    return True


def completed_fold_audit_is_valid(
    fold_audit_df,
    family,
    config,
    run_number,
):
    if fold_audit_df.empty:
        return False

    rows = fold_audit_df.loc[
        fold_audit_df["FamilyKey"].astype(str).eq(family)
        & fold_audit_df["ConfigurationID"].astype(str).eq(
            config["configuration_id"]
        )
        & pd.to_numeric(fold_audit_df["Run"], errors="coerce").eq(run_number)
        & fold_audit_df["ImplementationVersion"].astype(str).eq(
            DNN_IMPLEMENTATION_VERSION
        )
        & fold_audit_df["Status"].astype(str).eq("PASS")
    ].copy()

    if len(rows) != N_OUTER_FOLDS:
        return False

    folds = sorted(
        pd.to_numeric(rows["OuterFold"], errors="coerce")
        .dropna()
        .astype(int)
        .tolist()
    )

    return folds == list(range(1, N_OUTER_FOLDS + 1))


def completed_run_is_fully_persisted(
    run_metrics_df,
    fold_audit_df,
    family,
    config,
    run_number,
):
    return (
        completed_run_is_valid(
            run_metrics_df,
            family,
            config,
            run_number,
        )
        and completed_fold_audit_is_valid(
            fold_audit_df,
            family,
            config,
            run_number,
        )
    )


RUN_METRICS_DF = load_checkpoint_csv(
    RUN_METRICS_FILE,
    RUN_METRIC_COLUMNS,
)
FOLD_AUDIT_DF = load_checkpoint_csv(
    FOLD_AUDIT_FILE,
    FOLD_AUDIT_COLUMNS,
)

print("Existing valid/invalid run rows loaded:", len(RUN_METRICS_DF))
print("Existing fold-audit rows loaded       :", len(FOLD_AUDIT_DF))


Existing valid/invalid run rows loaded: 1460
Existing fold-audit rows loaded       : 7300


## 11. Mean, sample SD, and 95% CI — Step 15



In [14]:
MEAN_SD_COLUMNS = [
    "ConfigurationOrder",
    "ConfigurationID",
    "Method",
    "Backbone",
    "InternalMethod",
    "Ratio",
    "N_Runs",
]
for metric in METRIC_COLUMNS:
    MEAN_SD_COLUMNS.extend([
        f"{metric}_Mean",
        f"{metric}_SD",
    ])

CI_COLUMNS = [
    "Method",
    "Ratio",
    "Acc.",
    "Prec.",
    "Rec.",
    "F1",
    "ROC-AUC",
    "PR-AUC",
]

CI_METRIC_LABEL = {
    "Accuracy": "Acc.",
    "Precision": "Prec.",
    "Recall": "Rec.",
    "F1": "F1",
    "ROC_AUC": "ROC-AUC",
    "PR_AUC": "PR-AUC",
}


def format_mean_ci(mean, low, high):
    return f"{mean:.4f} [{low:.4f}, {high:.4f}]"


def build_family_summary_frames(run_metrics_df, family):
    family_runs = run_metrics_df.loc[
        run_metrics_df["FamilyKey"].astype(str).eq(family)
        & run_metrics_df["ImplementationVersion"].astype(str).eq(
            DNN_IMPLEMENTATION_VERSION
        )
        & run_metrics_df["Status"].astype(str).eq("PASS")
    ].copy()

    if not family_runs.empty:
        family_runs["ConfigurationOrder"] = pd.to_numeric(
            family_runs["ConfigurationOrder"],
            errors="raise",
        ).astype(int)
        family_runs["Run"] = pd.to_numeric(
            family_runs["Run"],
            errors="raise",
        ).astype(int)
        family_runs = family_runs.sort_values(
            ["ConfigurationOrder", "Run"]
        ).reset_index(drop=True)

    mean_sd_rows = []
    ci_rows = []

    for config in REPORT_CONFIGS:
        sub = family_runs.loc[
            family_runs["ConfigurationID"].astype(str).eq(
                config["configuration_id"]
            )
        ].copy()

        n_runs = len(sub)

        mean_sd_row = {
            "ConfigurationOrder": config["configuration_order"],
            "ConfigurationID": config["configuration_id"],
            "Method": config["method"],
            "Backbone": config["backbone"],
            "InternalMethod": config["internal_method"],
            "Ratio": config["ratio"],
            "N_Runs": n_runs,
        }

        ci_row = {
            "Method": config["method"],
            "Ratio": config["ratio"],
        }

        for metric in METRIC_COLUMNS:
            if n_runs > 0:
                values = pd.to_numeric(
                    sub[metric],
                    errors="raise",
                ).to_numpy(dtype=float)
                mean_value = float(np.mean(values))
            else:
                values = np.array([], dtype=float)
                mean_value = np.nan

            if n_runs >= 2:
                sd_value = float(np.std(values, ddof=1))
            else:
                sd_value = np.nan

            mean_sd_row[f"{metric}_Mean"] = mean_value
            mean_sd_row[f"{metric}_SD"] = sd_value

            display_col = CI_METRIC_LABEL[metric]

            if n_runs == RUNS_PER_CONFIGURATION:
                assert np.isfinite(sd_value)
                margin = (
                    T_CRITICAL_95_DF9
                    * sd_value
                    / math.sqrt(RUNS_PER_CONFIGURATION)
                )
                low = mean_value - margin
                high = mean_value + margin
                ci_row[display_col] = format_mean_ci(
                    mean_value,
                    low,
                    high,
                )
            else:
                ci_row[display_col] = ""

        mean_sd_rows.append(mean_sd_row)
        ci_rows.append(ci_row)

    mean_sd = pd.DataFrame(mean_sd_rows).reindex(columns=MEAN_SD_COLUMNS)
    ci = pd.DataFrame(ci_rows).reindex(columns=CI_COLUMNS)

    return family_runs, mean_sd, ci


## result workbook —  reporting structure




In [15]:
EXPECTED_WORKBOOK_SHEETS = []
for family, cfg in FAMILIES.items():
    EXPECTED_WORKBOOK_SHEETS.extend([
        f"{cfg['sheet']} Runs",
        f"{cfg['sheet']} MeanSD",
        f"{cfg['sheet']} CI",
    ])

assert len(EXPECTED_WORKBOOK_SHEETS) == 12
assert len(set(EXPECTED_WORKBOOK_SHEETS)) == 12

HEADER_FILL = PatternFill("solid", fgColor="1F4E78")
HEADER_FONT = Font(color="FFFFFF", bold=True)
THIN_GRAY = Side(style="thin", color="D9E1F2")
HEADER_BORDER = Border(bottom=THIN_GRAY)


def style_dataframe_sheet(ws, kind):
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    for cell in ws[1]:
        cell.fill = HEADER_FILL
        cell.font = HEADER_FONT
        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True,
        )
        cell.border = HEADER_BORDER

    ws.row_dimensions[1].height = 28

    header_map = {
        cell.value: cell.column
        for cell in ws[1]
    }

    if kind == "runs":
        widths = {
            "Family": 18,
            "Method": 27,
            "Backbone": 12,
            "InternalMethod": 18,
            "Ratio": 10,
            "Run": 8,
            "N_Folds": 9,
            "N_OOF": 12,
            "Accuracy": 12,
            "Precision": 12,
            "Recall": 12,
            "F1": 12,
            "ROC_AUC": 12,
            "PR_AUC": 12,
            "Status": 10,
            "CompletedAt": 20,
        }
    elif kind == "meansd":
        widths = {
            "Method": 27,
            "Backbone": 12,
            "InternalMethod": 18,
            "Ratio": 10,
            "N_Runs": 9,
        }
    else:
        widths = {
            "Method": 30,
            "Ratio": 10,
            "Acc.": 27,
            "Prec.": 27,
            "Rec.": 27,
            "F1": 27,
            "ROC-AUC": 27,
            "PR-AUC": 27,
        }

    for name, col_idx in header_map.items():
        width = widths.get(name, 16)
        ws.column_dimensions[get_column_letter(col_idx)].width = width

    # Numeric formatting for run and MeanSD sheets.
    if kind in {"runs", "meansd"}:
        metric_headers = set(METRIC_COLUMNS)
        if kind == "meansd":
            metric_headers = {
                f"{metric}_{suffix}"
                for metric in METRIC_COLUMNS
                for suffix in ["Mean", "SD"]
            }

        for header in metric_headers:
            if header in header_map:
                col_idx = header_map[header]
                for row in range(2, ws.max_row + 1):
                    ws.cell(row=row, column=col_idx).number_format = "0.0000"

    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(
                vertical="top",
                wrap_text=False,
            )


def write_results_workbook(run_metrics_df, raise_if_locked=False):
    if RESULTS_WORKBOOK_TEMP.exists():
        try:
            RESULTS_WORKBOOK_TEMP.unlink()
        except PermissionError as exc:
            if raise_if_locked:
                raise PermissionError(
                    f"Temporary results workbook is locked: {RESULTS_WORKBOOK_TEMP}"
                ) from exc
            print(
                "WARNING - temporary results workbook is locked; Excel refresh skipped."
            )
            return False

    with pd.ExcelWriter(
        RESULTS_WORKBOOK_TEMP,
        engine="openpyxl",
    ) as writer:
        for family, cfg in FAMILIES.items():
            family_runs, mean_sd, ci = build_family_summary_frames(
                run_metrics_df,
                family,
            )

            # Human-facing order first; technical audit fields remain available.
            run_sheet_columns = [
                "Family",
                "Method",
                "Backbone",
                "InternalMethod",
                "Ratio",
                "Run",
                "N_Folds",
                "N_OOF",
                "N_OOF_NonInteraction",
                "N_OOF_Interaction",
                "Accuracy",
                "Precision",
                "Recall",
                "F1",
                "ROC_AUC",
                "PR_AUC",
                "Run_Fit_Seconds",
                "Status",
                "CompletedAt",
                "ConfigurationOrder",
                "ConfigurationID",
                "RatioKey",
                "OOF_PairLabel_SHA256",
                "DNN_Seed_Policy",
                "ImplementationVersion",
                "FamilyKey",
            ]
            family_runs = family_runs.reindex(columns=run_sheet_columns)

            runs_sheet = f"{cfg['sheet']} Runs"
            meansd_sheet = f"{cfg['sheet']} MeanSD"
            ci_sheet = f"{cfg['sheet']} CI"

            family_runs.to_excel(
                writer,
                sheet_name=runs_sheet,
                index=False,
            )
            mean_sd.to_excel(
                writer,
                sheet_name=meansd_sheet,
                index=False,
            )
            ci.to_excel(
                writer,
                sheet_name=ci_sheet,
                index=False,
            )

            style_dataframe_sheet(writer.book[runs_sheet], "runs")
            style_dataframe_sheet(writer.book[meansd_sheet], "meansd")
            style_dataframe_sheet(writer.book[ci_sheet], "ci")

    # Verify that the temporary workbook is readable and has exactly 12 sheets.
    wb_check = load_workbook(
        RESULTS_WORKBOOK_TEMP,
        read_only=True,
        data_only=False,
    )
    actual_sheets = wb_check.sheetnames
    wb_check.close()

    assert actual_sheets == EXPECTED_WORKBOOK_SHEETS

    try:
        os.replace(
            RESULTS_WORKBOOK_TEMP,
            RESULTS_WORKBOOK,
        )
        return True
    except PermissionError as exc:
        message = (
            f"Results workbook is locked: {RESULTS_WORKBOOK}. "
            "Close it in Excel, disable File Explorer Preview Pane if needed, "
            "and rerun the final workbook cell. CSV checkpoints are safe."
        )
        if raise_if_locked:
            raise PermissionError(message) from exc
        print("WARNING -", message)
        return False


# Initialize/refresh the 12-sheet workbook before the long training stage.
workbook_written = write_results_workbook(
    RUN_METRICS_DF,
    raise_if_locked=False,
)

if workbook_written:
    print("PASS - 12-sheet results workbook initialized/refreshed:")
    print(RESULTS_WORKBOOK)
else:
    print("Workbook refresh skipped because the destination is locked.")


PASS - 12-sheet results workbook initialized/refreshed:
c:\Users\riskf\OneDrive\A-DTI2026\Stats\DNN_OOF_Final_Results.xlsx


## 13. Main training loop 


In [ ]:
def delete_completed_run_checkpoints(family, config, run_number):
    run_dir = run_checkpoint_dir(
        family,
        config,
        run_number,
    )
    if run_dir.exists():
        shutil.rmtree(run_dir)


def all_runs_complete_for_config(
    run_metrics_df,
    fold_audit_df,
    family,
    config,
):
    return all(
        completed_run_is_fully_persisted(
            run_metrics_df,
            fold_audit_df,
            family,
            config,
            run_number,
        )
        for run_number in range(1, RUNS_PER_CONFIGURATION + 1)
    )


def execute_steps_11_to_14():
    global RUN_METRICS_DF, FOLD_AUDIT_DF

    completed_new_runs = 0

    for family in FAMILIES_TO_RUN:
        assert family in FAMILIES

        print("\n" + "#" * 100)
        print(f"FAMILY: {FAMILIES[family]['display']}")
        print("#" * 100)

        for backbone in BACKBONES:
            backbone_configs = [
                config
                for config in REPORT_CONFIGS
                if config["backbone"] == backbone
            ]

            pending = [
                config
                for config in backbone_configs
                if not all_runs_complete_for_config(
                    RUN_METRICS_DF,
                    FOLD_AUDIT_DF,
                    family,
                    config,
                )
            ]

            if not pending:
                print(
                    f"{FAMILIES[family]['display']} | {backbone}: "
                    "all configured runs already complete; skipping cache build."
                )
                continue

            fold_cache = build_backbone_fold_cache(
                family,
                backbone,
            )

            for config in backbone_configs:
                print("\n" + "=" * 100)
                print(
                    f"{FAMILIES[family]['display']} | "
                    f"{config['method']} | {config['ratio']}"
                )
                print("=" * 100)

                if all_runs_complete_for_config(
                    RUN_METRICS_DF,
                    FOLD_AUDIT_DF,
                    family,
                    config,
                ):
                    print("All 10 run-level OOF results already complete. Resuming past configuration.")
                    continue

                for run_number in range(1, RUNS_PER_CONFIGURATION + 1):
                    if completed_run_is_fully_persisted(
                        RUN_METRICS_DF,
                        FOLD_AUDIT_DF,
                        family,
                        config,
                        run_number,
                    ):
                        print(f"  run {run_number:02d}/10: valid run-level checkpoint found; skipping.")
                        if DELETE_COMPLETED_RUN_FOLD_CHECKPOINTS:
                            delete_completed_run_checkpoints(
                                family,
                                config,
                                run_number,
                            )
                        continue

                    print(f"  run {run_number:02d}/10")

                    for outer_fold in range(1, N_OUTER_FOLDS + 1):
                        train_and_save_fold(
                            family,
                            config,
                            run_number,
                            outer_fold,
                            fold_cache[outer_fold],
                        )

                    # Step 13: pool the five outer-test folds into one OOF vector.
                    # Step 14: compute the six run-level metrics once on that vector.
                    run_row, fold_rows = assemble_run_oof_and_audit(
                        family,
                        config,
                        run_number,
                        fold_cache,
                    )

                    # Persist the run result BEFORE deleting temporary fold checkpoints.
                    RUN_METRICS_DF = upsert_rows(
                        RUN_METRICS_DF,
                        [run_row],
                        key_columns=[
                            "FamilyKey",
                            "ConfigurationID",
                            "Run",
                        ],
                        all_columns=RUN_METRIC_COLUMNS,
                        path=RUN_METRICS_FILE,
                    )

                    FOLD_AUDIT_DF = upsert_rows(
                        FOLD_AUDIT_DF,
                        fold_rows,
                        key_columns=[
                            "FamilyKey",
                            "ConfigurationID",
                            "Run",
                            "OuterFold",
                        ],
                        all_columns=FOLD_AUDIT_COLUMNS,
                        path=FOLD_AUDIT_FILE,
                    )

                    assert completed_run_is_fully_persisted(
                        RUN_METRICS_DF,
                        FOLD_AUDIT_DF,
                        family,
                        config,
                        run_number,
                    )

                    if DELETE_COMPLETED_RUN_FOLD_CHECKPOINTS:
                        delete_completed_run_checkpoints(
                            family,
                            config,
                            run_number,
                        )

                    completed_new_runs += 1

                    print(
                        f"  run {run_number:02d}/10: PASS OOF | "
                        f"N={run_row['N_OOF']:,} | "
                        f"F1={run_row['F1']:.4f} | "
                        f"ROC-AUC={run_row['ROC_AUC']:.4f} | "
                        f"PR-AUC={run_row['PR_AUC']:.4f}"
                    )

                    if REFRESH_EXCEL_AFTER_EACH_RUN:
                        write_results_workbook(
                            RUN_METRICS_DF,
                            raise_if_locked=False,
                        )

                assert all_runs_complete_for_config(
                    RUN_METRICS_DF,
                    FOLD_AUDIT_DF,
                    family,
                    config,
                )

                if REFRESH_EXCEL_AFTER_EACH_CONFIGURATION:
                    write_results_workbook(
                        RUN_METRICS_DF,
                        raise_if_locked=False,
                    )

                print("Configuration complete: 10 / 10 full-family OOF runs.")

            del fold_cache
            clear_session()
            gc.collect()

    print("\n" + "=" * 100)
    print("Steps 11-14 execution finished.")
    print("New completed run-level results:", completed_new_runs)
    print("Run checkpoint rows:", len(RUN_METRICS_DF))
    print("Fold audit rows    :", len(FOLD_AUDIT_DF))
    print("=" * 100)


execute_steps_11_to_14()



####################################################################################################
FAMILY: Enzyme
####################################################################################################
Preparing fixed scaled fold cache: Enzyme | MSCMF
  fold 1: train=189,107, validation=47,277, test=59,096
  fold 2: train=189,107, validation=47,277, test=59,096
  fold 3: train=189,107, validation=47,277, test=59,096
  fold 4: train=189,107, validation=47,277, test=59,096
  fold 5: train=189,107, validation=47,277, test=59,096

Enzyme | MSCMF (No DA) | No aug.
  run 01/10: valid run-level checkpoint found; skipping.
  run 02/10


C:\Users\riskf\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


    fold 1: PASS | train=189,107 | validation=47,277 | test=59,096 | 45.5s


C:\Users\riskf\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


    fold 2: PASS | train=189,107 | validation=47,277 | test=59,096 | 44.5s


...

C:\Users\riskf\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


    fold 2: PASS | train=52,944 | validation=6,855 | test=8,568 | 25.4s


C:\Users\riskf\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


    fold 3: PASS | train=52,944 | validation=6,855 | test=8,568 | 15.8s


Exception ignored in: <function WeakKeyDictionary.__init__.<locals>.remove at 0x0000025F84125B20>
Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\Lib\weakref.py", line 370, in remove
    self = selfref()
           ^^^^^^^^^
KeyboardInterrupt: 
C:\Users\riskf\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [12]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
STATS_DIR = PROJECT_ROOT / "Stats"

# ============================================================
# USE THE FINAL FOUR-FAMILY MERGED CHECKPOINTS
# ============================================================

RUN_METRICS_FILE = (
    STATS_DIR /
    "dnn_11_17_run_metrics_checkpoint_MERGED.csv"
)

FOLD_AUDIT_FILE = (
    STATS_DIR /
    "dnn_11_17_fold_audit_checkpoint_MERGED.csv"
)

assert RUN_METRICS_FILE.exists(), (
    f"Missing merged run file: {RUN_METRICS_FILE}"
)

assert FOLD_AUDIT_FILE.exists(), (
    f"Missing merged fold file: {FOLD_AUDIT_FILE}"
)

# ============================================================
# RELOAD INTO THE NOTEBOOK DATAFRAMES
# ============================================================

RUN_METRICS_DF = load_checkpoint_csv(
    RUN_METRICS_FILE,
    RUN_METRIC_COLUMNS,
)

FOLD_AUDIT_DF = load_checkpoint_csv(
    FOLD_AUDIT_FILE,
    FOLD_AUDIT_COLUMNS,
)

# ============================================================
# BASIC FINAL CHECK
# ============================================================

assert len(RUN_METRICS_DF) == 2920, (
    f"Expected 2,920 run rows, found {len(RUN_METRICS_DF):,}"
)

assert len(FOLD_AUDIT_DF) == 14600, (
    f"Expected 14,600 fold rows, found {len(FOLD_AUDIT_DF):,}"
)

expected_families = {
    "enzyme",
    "gpcr",
    "ion_channel",
    "nuclear_receptor",
}

assert set(
    RUN_METRICS_DF["FamilyKey"].astype(str).unique()
) == expected_families

assert set(
    FOLD_AUDIT_DF["FamilyKey"].astype(str).unique()
) == expected_families

print("=" * 100)
print("PASS - MERGED FOUR-FAMILY RESULTS LOADED")
print("=" * 100)

print("Run rows :", f"{len(RUN_METRICS_DF):,}")
print("Fold rows:", f"{len(FOLD_AUDIT_DF):,}")

print("\nRun rows by family:")
print(
    RUN_METRICS_DF
    .groupby("FamilyKey")
    .size()
)

print("\nFold rows by family:")
print(
    FOLD_AUDIT_DF
    .groupby("FamilyKey")
    .size()
)

print("=" * 100)

PASS - MERGED FOUR-FAMILY RESULTS LOADED
Run rows : 2,920
Fold rows: 14,600

Run rows by family:
FamilyKey
enzyme              730
gpcr                730
ion_channel         730
nuclear_receptor    730
dtype: int64

Fold rows by family:
FamilyKey
enzyme              3650
gpcr                3650
ion_channel         3650
nuclear_receptor    3650
dtype: int64


## build final means, sample SDs, and 95% CIs


In [16]:
FAMILY_SUMMARIES = {}

for family in FAMILIES:
    family_runs, mean_sd, ci = build_family_summary_frames(
        RUN_METRICS_DF,
        family,
    )

    FAMILY_SUMMARIES[family] = {
        "runs": family_runs,
        "mean_sd": mean_sd,
        "ci": ci,
    }

    complete_configs = int(
        (mean_sd["N_Runs"] == RUNS_PER_CONFIGURATION).sum()
    )

    print(
        f"{FAMILIES[family]['display']}: "
        f"run rows={len(family_runs):,} | "
        f"complete configurations={complete_configs}/{len(REPORT_CONFIGS)}"
    )


Enzyme: run rows=730 | complete configurations=73/73
GPCR: run rows=730 | complete configurations=73/73
Ion Channel: run rows=730 | complete configurations=73/73
Nuclear Receptor: run rows=730 | complete configurations=73/73


##  strict completeness and OOF audit



In [17]:
def validate_complete_results(run_metrics_df, fold_audit_df):
    validation_rows = []

    def add_check(name, passed, observed, expected):
        validation_rows.append({
            "Check": name,
            "Pass": bool(passed),
            "Observed": str(observed),
            "Expected": str(expected),
        })

    current_runs = run_metrics_df.loc[
        run_metrics_df["ImplementationVersion"].astype(str).eq(
            DNN_IMPLEMENTATION_VERSION
        )
        & run_metrics_df["Status"].astype(str).eq("PASS")
    ].copy()

    current_folds = fold_audit_df.loc[
        fold_audit_df["ImplementationVersion"].astype(str).eq(
            DNN_IMPLEMENTATION_VERSION
        )
        & fold_audit_df["Status"].astype(str).eq("PASS")
    ].copy()

    add_check(
        "Total run-level rows",
        len(current_runs) == EXPECTED_TOTAL_RUN_ROWS,
        len(current_runs),
        EXPECTED_TOTAL_RUN_ROWS,
    )

    run_key_cols = ["FamilyKey", "ConfigurationID", "Run"]
    run_duplicates = current_runs.duplicated(run_key_cols).sum()
    add_check(
        "Duplicate run keys",
        run_duplicates == 0,
        run_duplicates,
        0,
    )

    for family in FAMILIES:
        family_runs = current_runs.loc[
            current_runs["FamilyKey"].astype(str).eq(family)
        ].copy()

        add_check(
            f"{FAMILIES[family]['display']} run rows",
            len(family_runs) == EXPECTED_RUNS_PER_FAMILY,
            len(family_runs),
            EXPECTED_RUNS_PER_FAMILY,
        )

        for config in REPORT_CONFIGS:
            sub = family_runs.loc[
                family_runs["ConfigurationID"].astype(str).eq(
                    config["configuration_id"]
                )
            ]

            runs = sorted(
                pd.to_numeric(sub["Run"], errors="coerce")
                .dropna()
                .astype(int)
                .tolist()
            )

            add_check(
                f"{FAMILIES[family]['display']} | {config['configuration_id']} | runs",
                runs == list(range(1, RUNS_PER_CONFIGURATION + 1)),
                runs,
                list(range(1, RUNS_PER_CONFIGURATION + 1)),
            )

        n_oof = pd.to_numeric(family_runs["N_OOF"], errors="coerce")
        add_check(
            f"{FAMILIES[family]['display']} OOF counts",
            (n_oof == FAMILY_REFERENCE[family]["n_total"]).all(),
            sorted(n_oof.dropna().unique().tolist()),
            FAMILY_REFERENCE[family]["n_total"],
        )

        shas = set(family_runs["OOF_PairLabel_SHA256"].astype(str))
        add_check(
            f"{FAMILIES[family]['display']} OOF pair/label hash",
            shas == {FAMILY_REFERENCE[family]["pair_label_sha256"]},
            shas,
            FAMILY_REFERENCE[family]["pair_label_sha256"],
        )

    n_folds = pd.to_numeric(current_runs["N_Folds"], errors="coerce")
    add_check(
        "Run-level N_Folds",
        (n_folds == N_OUTER_FOLDS).all(),
        sorted(n_folds.dropna().unique().tolist()),
        N_OUTER_FOLDS,
    )

    metric_matrix = current_runs[METRIC_COLUMNS].apply(
        pd.to_numeric,
        errors="coerce",
    ).to_numpy(dtype=float)

    add_check(
        "Finite run-level metrics",
        np.isfinite(metric_matrix).all(),
        int(np.isfinite(metric_matrix).sum()),
        int(metric_matrix.size),
    )

    in_range = (
        np.isfinite(metric_matrix).all()
        and ((metric_matrix >= -1e-12) & (metric_matrix <= 1 + 1e-12)).all()
    )
    add_check(
        "Run-level metrics in [0,1]",
        in_range,
        "all" if in_range else "one or more outside range",
        "all",
    )

    add_check(
        "Total fold-audit rows",
        len(current_folds) == EXPECTED_TOTAL_DNN_FITS,
        len(current_folds),
        EXPECTED_TOTAL_DNN_FITS,
    )

    fold_key_cols = [
        "FamilyKey",
        "ConfigurationID",
        "Run",
        "OuterFold",
    ]
    fold_duplicates = current_folds.duplicated(fold_key_cols).sum()
    add_check(
        "Duplicate fold-audit keys",
        fold_duplicates == 0,
        fold_duplicates,
        0,
    )

    if len(current_folds):
        group_counts = (
            current_folds
            .groupby(["FamilyKey", "ConfigurationID", "Run"])["OuterFold"]
            .agg(lambda x: sorted(pd.to_numeric(x).astype(int).tolist()))
        )
        bad_fold_groups = int(
            sum(
                folds != list(range(1, N_OUTER_FOLDS + 1))
                for folds in group_counts
            )
        )
    else:
        bad_fold_groups = EXPECTED_TOTAL_RUN_ROWS

    add_check(
        "Every completed run has folds 1-5 exactly once",
        bad_fold_groups == 0,
        bad_fold_groups,
        0,
    )

    validation = pd.DataFrame(validation_rows)

    # Save Step 16 audit separately; the result workbook remains exactly 12 sheets.
    temp_validation = VALIDATION_WORKBOOK.with_name(
        VALIDATION_WORKBOOK.stem + "_TEMP.xlsx"
    )
    if temp_validation.exists():
        temp_validation.unlink()

    with pd.ExcelWriter(temp_validation, engine="openpyxl") as writer:
        validation.to_excel(
            writer,
            sheet_name="Validation",
            index=False,
        )

    try:
        os.replace(temp_validation, VALIDATION_WORKBOOK)
    except PermissionError as exc:
        raise PermissionError(
            f"Validation workbook is locked: {VALIDATION_WORKBOOK}"
        ) from exc

    failed = validation.loc[~validation["Pass"]]

    display(validation)

    if not failed.empty:
        raise AssertionError(
            f"Step 16 FAILED: {len(failed)} validation check(s) did not pass. "
            f"See {VALIDATION_WORKBOOK}."
        )

    print("PASS - Step 16 complete-results audit passed.")
    return validation


FINAL_VALIDATION = validate_complete_results(
    RUN_METRICS_DF,
    FOLD_AUDIT_DF,
)


,Check,Pass,Observed,Expected
0,Total run-level rows,True,2920,2920
1,Duplicate run keys,True,0,0
2,Enzyme run rows,True,730,730
3,Enzyme | MSCMF__NoDA__NO_DA | runs,True,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]"
4,Enzyme | MSCMF__WGANGP__50_50 | runs,True,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]"
...,...,...,...,...
307,Finite run-level metrics,True,17520,17520
308,"Run-level metrics in [0,1]",True,all,all
309,Total fold-audit rows,True,14600,14600
310,Duplicate fold-audit keys,True,0,0


PASS - Step 16 complete-results audit passed.


## generate the final workbook


In [18]:
# Force a final workbook rebuild. At this stage the output must not be open in Excel.
write_results_workbook(
    RUN_METRICS_DF,
    raise_if_locked=True,
)

# Final workbook validation.
wb_final = load_workbook(
    RESULTS_WORKBOOK,
    read_only=True,
    data_only=False,
)
final_sheet_names = wb_final.sheetnames
wb_final.close()

assert final_sheet_names == EXPECTED_WORKBOOK_SHEETS

# Verify each family report has the expected number of rows.
for family, cfg in FAMILIES.items():
    runs, mean_sd, ci = build_family_summary_frames(
        RUN_METRICS_DF,
        family,
    )

    assert len(runs) == EXPECTED_RUNS_PER_FAMILY
    assert len(mean_sd) == EXPECTED_CONFIGS_PER_FAMILY
    assert len(ci) == EXPECTED_CONFIGS_PER_FAMILY
    assert (mean_sd["N_Runs"] == RUNS_PER_CONFIGURATION).all()
    assert (
        ci[["Acc.", "Prec.", "Rec.", "F1", "ROC-AUC", "PR-AUC"]]
        .ne("")
        .all()
        .all()
    )

manifest = {
    "implementation_version": DNN_IMPLEMENTATION_VERSION,
    "steps": [
        "11 - fixed DNN training",
        "12 - untouched outer-test prediction",
        "13 - five-fold OOF assembly",
        "14 - run-level OOF metrics",
        "15 - mean, sample SD, and 95% CI over 10 runs",
        "16 - complete-results validation",
        "17 - family result tables",
    ],
    "families": list(FAMILIES.keys()),
    "outer_folds": N_OUTER_FOLDS,
    "runs_per_configuration": RUNS_PER_CONFIGURATION,
    "configurations_per_family": EXPECTED_CONFIGS_PER_FAMILY,
    "run_level_rows_per_family": EXPECTED_RUNS_PER_FAMILY,
    "total_run_level_rows": EXPECTED_TOTAL_RUN_ROWS,
    "total_dnn_fits": EXPECTED_TOTAL_DNN_FITS,
    "baseline": {
        "method": "MSCMF (No DA)",
        "backbone": "MSCMF",
        "ratio": "natural inner-training prevalence",
        "runs": RUNS_PER_CONFIGURATION,
    },
    "augmentation_configurations": [
        {
            "backbone": backbone,
            "method": method,
            "ratios": [RATIO_DISPLAY[key] for key in TARGET_RATIOS],
        }
        for backbone, method in REPORT_METHOD_ORDER
    ],
    "dnn": {
        "input_dimension": 100,
        "hidden_layers": [512, 256, 128],
        "hidden_activation": "ReLU",
        "dropout": 0.1,
        "dropout_position": "after first hidden layer",
        "output_units": 2,
        "output_activation": "Sigmoid",
        "optimizer": "Adam",
        "learning_rate": 0.001,
        "beta_1": 0.9,
        "beta_2": 0.999,
        "epsilon": 1e-7,
        "loss": "binary_crossentropy",
        "epochs": 12,
        "batch_sizes": {
            family: FAMILIES[family]["batch_size"]
            for family in FAMILIES
        },
    },
    "scaler": {
        "type": "StandardScaler",
        "fit_scope": "real inner-training observations only, per family/fold/backbone",
        "validation_used_for_fit": False,
        "outer_test_used_for_fit": False,
    },
    "seed_policy": {
        "data_partitions": "fixed and reused from Steps 05-06",
        "dnn_run_seed": "not fixed and not saved",
        "seed_functions_called_by_this_notebook": False,
        "interpretation": "10 stochastic DNN training runs under fixed data partitions",
    },
    "metrics": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC",
        "PR-AUC",
    ],
    "metric_scope": "one pooled full-family OOF prediction vector per run",
    "confidence_interval": {
        "n": 10,
        "sample_sd_ddof": 1,
        "distribution": "Student t",
        "df": 9,
        "t_critical_0.975": T_CRITICAL_95_DF9,
    },
    "results_workbook": str(RESULTS_WORKBOOK.relative_to(PROJECT_ROOT)),
    "results_sheets": EXPECTED_WORKBOOK_SHEETS,
    "run_metrics_checkpoint": str(RUN_METRICS_FILE.relative_to(PROJECT_ROOT)),
    "fold_audit_checkpoint": str(FOLD_AUDIT_FILE.relative_to(PROJECT_ROOT)),
    "validation_workbook": str(VALIDATION_WORKBOOK.relative_to(PROJECT_ROOT)),
    "step_09_10_manifest_sha256": file_sha256(AUGMENTATION_MANIFEST_FILE),
    "created_at": datetime.now().isoformat(timespec="seconds"),
}

DNN_MANIFEST_FILE.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

print("\n" + "=" * 100)
print("PASS - Steps 11-17 completed successfully.")
print("PASS - 10 stochastic full-family OOF runs per configuration.")
print("PASS - 73 configurations per family: 18 x 4 ratios + MSCMF No-DA baseline.")
print("PASS - 2,920 run-level OOF result rows.")
print("PASS - 14,600 DNN fold fits audited.")
print("PASS - Final results workbook contains exactly 12 sheets.")
print("Results workbook  :", RESULTS_WORKBOOK)
print("Validation workbook:", VALIDATION_WORKBOOK)
print("Manifest          :", DNN_MANIFEST_FILE)
print("=" * 100)



PASS - Steps 11-17 completed successfully.
PASS - 10 stochastic full-family OOF runs per configuration.
PASS - 73 configurations per family: 18 x 4 ratios + MSCMF No-DA baseline.
PASS - 2,920 run-level OOF result rows.
PASS - 14,600 DNN fold fits audited.
PASS - Final results workbook contains exactly 12 sheets.
Results workbook  : c:\Users\riskf\OneDrive\A-DTI2026\Stats\DNN_OOF_Final_Results.xlsx
Validation workbook: c:\Users\riskf\OneDrive\A-DTI2026\Stats\DNN_11_17_Validation.xlsx
Manifest          : c:\Users\riskf\OneDrive\A-DTI2026\data\split\dnn_11_17_manifest.json
